# Building a Multi-Agent Research Assistant with OpenAI Agents SDK and Olostep

This notebook builds a beginner-friendly multi-agent research assistant using:

- **OpenAI Agents SDK** for orchestration, specialist tools, direct retrieval tools, and structured outputs.
- **Olostep Python SDK** for Answer API, Search, Search with Scrape, and Scrape.

The final output is a **proper Markdown research report** for any user-provided research question, not just a short answer.


## Setup

Run this once. The `-U` matters because older `olostep` versions did not include `client.searches`.


In [12]:
# %pip install -q -U openai-agents olostep python-dotenv pydantic

## Environment Variables

Create a `.env` file next to this notebook:

```bash
OPENAI_API_KEY=your_openai_api_key
OLOSTEP_API_KEY=your_olostep_api_key
```


In [13]:
import importlib.metadata
import json
import os
import textwrap
import warnings
from datetime import datetime
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from olostep import Olostep
from pydantic import BaseModel, Field

from agents import (
    Agent,
    Runner,
    custom_span,
    flush_traces,
    function_tool,
    gen_trace_id,
    trace,
)

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OLOSTEP_API_KEY = os.getenv("OLOSTEP_API_KEY")

## Olostep SDK Search With Scrape Demo

This standalone cell uses the exact SDK style requested.


In [14]:
client = Olostep(api_key=OLOSTEP_API_KEY)

search = client.searches.create(
    query="What are the most important recent developments in AI agents for business research?",
    limit=5,
    scrape_options={"formats": ["markdown"], "timeout": 25},
)

for link in search.links:
    print(link["url"], "-", len(link.get("markdown_content") or ""), "chars")


https://www.bcg.com/capabilities/artificial-intelligence/ai-agents - 28638 chars
https://company.g2.com/news/2025-ai-agent-report - 14118 chars
https://www.grandviewresearch.com/industry-analysis/ai-agents-market-report - 28563 chars
https://mitsloan.mit.edu/ideas-made-to-matter/4-new-studies-about-agentic-ai-mit-initiative-digital-economy - 21678 chars
https://www.capgemini.com/insights/research-library/ai-agents/ - 49027 chars


## Helpers

These helpers keep SDK responses compact and JSON-friendly for agent tool outputs.

Each Olostep helper also creates an OpenAI Agents trace span, so the trace shows which retrieval endpoint ran.


In [15]:
class OlostepError(RuntimeError):
    """Raised when an Olostep SDK request fails."""


def require_olostep_key() -> str:
    if not OLOSTEP_API_KEY:
        raise OlostepError(
            "OLOSTEP_API_KEY is not set. Add it to .env and rerun the setup cell."
        )
    return OLOSTEP_API_KEY


def get_olostep_client() -> Olostep:
    return Olostep(api_key=require_olostep_key())


def sdk_result_to_dict(result: Any) -> dict[str, Any]:
    if hasattr(result, "model_dump"):
        return result.model_dump()
    if hasattr(result, "__dict__"):
        return {
            key: value for key, value in vars(result).items() if not key.startswith("_")
        }
    return {"value": str(result)}


def compact_json(data: Any, max_chars: int = 8000) -> str:
    text = json.dumps(data, ensure_ascii=False, indent=2, default=str)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n... [truncated]"


def current_date_context() -> str:
    return datetime.now().strftime("%B %d, %Y")


def current_year_context() -> str:
    return str(datetime.now().year)


def normalize_search_links(
    links: list[dict[str, Any]], limit: int = 8
) -> list[dict[str, Any]]:
    rows = []
    for link in links[:limit]:
        markdown = link.get("markdown_content") or ""
        rows.append(
            {
                "title": link.get("title") or "Untitled",
                "url": link.get("url") or "",
                "description": link.get("description") or "",
                "markdown_chars": len(markdown),
                "markdown_preview": markdown[:1500] if markdown else "",
            }
        )
    return rows


## Pydantic Structured Outputs

These models make the judge output and final research report predictable.


In [16]:
class Judgment(BaseModel):
    is_good_enough: bool = Field(
        description="Whether the answer is sufficient for the user query, meaning score >= 0.85."
    )
    score: float = Field(ge=0, le=1, description="Quality score from 0 to 1.")
    reason: str = Field(description="Short explanation of the decision.")
    missing_information: list[str] = Field(
        default_factory=list, description="Important gaps to fix."
    )


class MarkdownResearchReport(BaseModel):
    title: str = Field(description="Research report title.")
    executive_summary: str = Field(description="Short answer-first summary.")
    key_findings: list[str] = Field(description="Most important findings.")
    markdown_report: str = Field(
        description="Complete Markdown report with polished headings, clear analysis, reader-friendly structure, and citations."
    )
    citations: list[str] = Field(
        default_factory=list, description="Source URLs used in the report."
    )
    confidence: str = Field(description="Low, medium, or high confidence.")
    method_used: str = Field(description="Retrieval path used by the manager agent.")


## Olostep Tool Functions

These are the four requested functions. Each one is exposed as an OpenAI Agents SDK `function_tool`.


In [17]:
@function_tool
async def answer_query(query: str) -> str:
    """Answer a natural-language research query using Olostep Answer API."""
    try:
        with custom_span("olostep.answer_query", {"query": query}):
            result = get_olostep_client().answers.create(task=query)
            return compact_json(sdk_result_to_dict(result))

    except Exception as exc:
        raise OlostepError(f"Olostep Answer API failed: {exc}") from exc


@function_tool
async def search_web(query: str, limit: int = 8) -> str:
    """Search the web using Olostep Search and return normalized results."""
    try:
        with custom_span("olostep.search_web", {"query": query, "limit": limit}):
            search = get_olostep_client().searches.create(
                query=query,
                limit=limit,
            )

            data = sdk_result_to_dict(search)

            return compact_json(
                {
                    "query": query,
                    "results": normalize_search_links(
                        data.get("links", []),
                        limit=limit,
                    ),
                    "raw": data,
                }
            )

    except Exception as exc:
        raise OlostepError(f"Olostep Search API failed: {exc}") from exc


@function_tool
async def search_with_scrape(query: str, limit: int = 5) -> str:
    """Search the web and scrape each returned link using Olostep Search with Scrape."""
    scrape_options = {
        "formats": ["markdown"],
        "timeout": 25,
    }

    try:
        with custom_span(
            "olostep.search_with_scrape",
            {
                "query": query,
                "limit": limit,
                "scrape_options": scrape_options,
            },
        ):
            search = get_olostep_client().searches.create(
                query=query,
                limit=limit,
                scrape_options=scrape_options,
            )

            data = sdk_result_to_dict(search)

            return compact_json(
                {
                    "query": query,
                    "results": normalize_search_links(
                        data.get("links", []),
                        limit=limit,
                    ),
                    "raw": data,
                },
                max_chars=12000,
            )

    except Exception as exc:
        raise OlostepError(f"Olostep Search with Scrape failed: {exc}") from exc


@function_tool
async def scrape_url(url: str) -> str:
    """Scrape one URL with Olostep and return compact page content."""
    try:
        with custom_span(
            "olostep.scrape_url",
            {
                "url": url,
                "formats": ["markdown"],
            },
        ):
            scrape = get_olostep_client().scrapes.create(
                url=url,
                formats=["markdown"],
            )

            return compact_json(
                {
                    "url": url,
                    "scrape": sdk_result_to_dict(scrape),
                },
                max_chars=10000,
            )

    except Exception as exc:
        raise OlostepError(f"Olostep Scrape API failed: {exc}") from exc

## Specialist Agents

The manager uses the judge and analyst as specialist tools, while calling Olostep retrieval tools directly.

This keeps the research path explicit: answer, judge, search with scrape, judge, targeted search, selected scrapes, then analysis.


In [18]:
MODEL = "gpt-5.4-mini"

judge_agent = Agent(
    name="Judge agent",
    model=MODEL,
    instructions=(
        "You judge whether the provided answer is good enough for the original research question. "
        "Reward direct, specific, source-backed answers. Reject vague, stale, or unsupported answers. "
        "Be strict: is_good_enough must be true only when score >= 0.85 and the evidence directly answers "
        "the question with concrete source content, topic-specific detail, and appropriate recency. "
        "For current events, product status, policies, pricing, or factual claims that may change, require recent "
        "primary or highly reputable sources. Do not mark evidence sufficient if any critical gap remains. "
        "Calibrate scores this way: 0.85-1.0 means sufficient to stop with strong source support and no critical gaps; "
        "0.75-0.84 means strong but still missing one important source, detail, recency check, or coverage area; "
        "0.50-0.74 means relevant partial evidence that needs more research; 0.25-0.49 means thin, vague, stale, "
        "or weakly related evidence; below 0.25 is only for empty, unusable, or mostly unrelated evidence. "
        "Do not mark evidence sufficient just because it is plausible or directionally correct. "
        "Return only the structured judgment."
    ),
    output_type=Judgment,
)

analyst_agent = Agent(
    name="Analyst agent",
    model=MODEL,
    instructions=(
        "You write a proper Markdown research report from the evidence. "
        "Write for a professional reader who wants a clear, polished research brief on any topic. "
        "Adapt the report to the user's question. The markdown_report must be substantial, easy to scan, and use these general sections only: "
        "Executive Summary, Key Findings, Context, Evidence Review, Detailed Analysis, Implications, Source Notes, and References. "
        "If the topic is event-driven, include timeline details inside Context or Detailed Analysis instead of adding a separate Timeline section. "
        "If the topic is comparative, include a compact comparison table inside Detailed Analysis. "
        "Do not include sections titled Limitations, Next Steps, Recommendations, or Action Items. "
        "Avoid bare caveats like 'I relied on...'. Instead, integrate source quality naturally in Source Notes. "
        "Use short paragraphs, bullets where helpful, and citations as Markdown links or URL bullets. "
        "Add enough context that a non-expert reader understands the issue, why it matters, and what evidence supports it. "
        "Do not use emoji, return-arrow symbols, backlink icons, or decorative icons anywhere in the report. "
        "In References, list only plain Markdown bullets or numbered items with the source name and URL. "
        "Return only the structured report."
    ),
    output_type=MarkdownResearchReport,
)


## Manager Agent As Orchestrator

Here the manager is given the judge, analyst, and direct Olostep retrieval tools. The notebook will run only this manager agent for the real research workflow.


In [19]:
judge_tool = judge_agent.as_tool(
    tool_name="judge_answer_quality",
    tool_description="Judge whether an answer or evidence is good enough for the original research question.",
)

analyst_tool = analyst_agent.as_tool(
    tool_name="write_markdown_research_report",
    tool_description="Write the final structured Markdown research report from the gathered evidence.",
)

manager_agent = Agent(
    name="Manager research agent",
    model=MODEL,
    instructions=(
        f"Current date: {current_date_context()}\n"
        f"Current year: {current_year_context()}\n\n"
        "You are the orchestrator for a multi-agent research assistant. You must manage the workflow, "
        "not answer from your own memory. Follow this policy exactly:\n"
        "1. Always call answer_query first to get a simple initial answer for the user's question.\n"
        "2. Immediately call judge_answer_quality on the original question plus the answer_query result. "
        "If the judge returns is_good_enough=true and score >= 0.85, stop researching and call "
        "write_markdown_research_report with the question, answer result, and judgment.\n"
        "3. If the first judgment is weak, call search_with_scrape for the original question. "
        "Immediately call judge_answer_quality again on the original question plus the answer_query result, "
        "first judgment, and search_with_scrape result. If this second judge returns is_good_enough=true "
        "and score >= 0.85, stop researching and call write_markdown_research_report with all evidence.\n"
        "4. If the second judgment is still weak, do not call the judge again. Run multiple targeted "
        "search_web calls first, using the judge's missing_information to form the searches. Inspect the "
        "search results, choose at least the top 3 relevant source URLs most likely to answer the missing "
        "points, then call scrape_url on each of those top 3 pages. Scrape more than 3 only if clearly needed.\n"
        "5. Call write_markdown_research_report exactly once at the end, using every answer, judgment, "
        "search result, and scraped page. The analyst must produce the final MarkdownResearchReport.\n"
        "6. Return only the final MarkdownResearchReport. Do not return a casual chat answer, tool transcript, or plan."
    ),
    tools=[answer_query, judge_tool, search_with_scrape, search_web, scrape_url, analyst_tool],
    output_type=MarkdownResearchReport,
)


## Run The Orchestrated Research Assistant With Tracing

This is now a single top-level manager run. The manager decides when to call the judge, analyst, and direct Olostep retrieval tools.


In [20]:
def openai_trace_url(trace_id: str) -> str:
    return f"https://platform.openai.com/logs/trace?trace_id={trace_id}"


async def run_research_assistant(query: str) -> MarkdownResearchReport:
    if not OPENAI_API_KEY:
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Add it to .env and rerun the setup cell."
        )
    require_olostep_key()

    trace_id = gen_trace_id()
    print("OpenAI trace ID:", trace_id)
    print("OpenAI trace URL:", openai_trace_url(trace_id))

    current_date = current_date_context()
    current_year = current_year_context()
    prompt = f"""
Current date:
{current_date}

Current year:
{current_year}

Research question:
{query}

Return a polished, reader-friendly Markdown research report with substantial detail for the user's specific question. Follow the required workflow exactly:
- Use answer_query first for a simple initial answer.
- Use the judge agent immediately after the simple answer to decide whether to stop or continue.
- If the first judge says the answer is not sufficient, run search_with_scrape.
- Use the judge agent immediately after search_with_scrape to decide whether to stop or continue.
- If the second judge still says the evidence is weak, do not judge again. Run multiple targeted search_web calls, choose at least the top 3 relevant source URLs from the search results, and scrape those top 3 pages for context.
- Analyst agent writes the final Markdown report from all answer, judge, search, and scrape evidence. Do not include Limitations or Next Steps sections.
"""
    with trace(
        workflow_name="multi_agent_research_assistant_olostep",
        trace_id=trace_id,
        metadata={
            "query": query,
            "notebook": "multi_agent_research_assistant_openai_agents_olostep",
        },
    ):
        with custom_span("manager.run", {"query": query}):
            result = await Runner.run(manager_agent, prompt, max_turns=30)

    flush_traces()
    print(
        "Trace flushed. Open the URL above to inspect manager, specialist agents, tools, and Olostep spans."
    )
    return result.final_output


## Example Query

In [21]:
example_query = "What's going on with OpenAI's Sora shutting down?"

report = await run_research_assistant(example_query)
display(Markdown(report.markdown_report))


OpenAI trace ID: trace_45f3333363da420dbcefc8bb8819224c
OpenAI trace URL: https://platform.openai.com/logs/trace?trace_id=trace_45f3333363da420dbcefc8bb8819224c
Trace flushed. Open the URL above to inspect manager, specialist agents, tools, and Olostep spans.


## Executive Summary

OpenAI is shutting down Sora as a product line in phases. According to OpenAI’s Help Center, the Sora web and app experiences were discontinued on April 26, 2026, and the Sora API is scheduled to be discontinued on September 24, 2026. OpenAI also posted guidance for exporting content and handling account-related issues, which suggests a managed wind-down rather than an emergency outage. [OpenAI Help Center](https://help.openai.com/)

Independent reporting from the Los Angeles Times and The New York Times corroborates the shutdown and frames it as part of a broader strategic shift. The coverage connects the move to business and partnership realities, including reporting about a Disney-related arrangement. [Los Angeles Times](https://www.latimes.com/) [The New York Times](https://www.nytimes.com/)

## What’s happening

Sora is OpenAI’s text-to-video generation product. The current issue is not speculation: OpenAI has announced a staged retirement of the service.

The timing is important:

| Milestone | Date | Meaning |
|---|---:|---|
| Web/app shutdown | Apr. 26, 2026 | Consumer access ended |
| API shutdown | Sep. 24, 2026 | Developer access ends later |

That split timeline usually means a company is reducing product surface area while giving users and developers time to migrate or export data. [OpenAI Help Center](https://help.openai.com/)

## Why it’s being shut down

OpenAI’s own notice focuses on the discontinuation process and what users should do next. Outside reporting adds the broader context:

- the product’s commercial viability was in question,
- the company appears to be refocusing resources,
- and a Disney-related partnership backdrop is part of the story. [Los Angeles Times](https://www.latimes.com/) [The New York Times](https://www.nytimes.com/)

So the best read is: this is a strategic wind-down, not a surprise technical failure.

## What users should take from it

- Sora is no longer a stable consumer product.
- If you have content in Sora, export it as soon as possible.
- If you depend on the API, the key deadline is September 24, 2026.

## Bottom line

OpenAI is discontinuing Sora in an orderly, phased shutdown. The service is already gone from the web/app side, and the API has a later retirement date. The move appears to reflect strategic and business considerations more than any single technical issue.

## References

- OpenAI Help Center — https://help.openai.com/en/articles/20001152-what-to-know-about-the-sora-discontinuation
- Los Angeles Times — https://www.latimes.com/entertainment-arts/business/story/2026-03-24/openai-will-shut-down-sora-why-what-to-know
- The New York Times — https://www.nytimes.com/2026/03/24/technology/openai-shutting-down-sora.html

![image_1.png](image_1.png)

## Save A Markdown Report

After running the live example, you can save the report to a `.md` file.


In [22]:
# After running the live example above:
output_path = "research_report.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(report.markdown_report)
print(f"Saved {output_path}")


Saved research_report.md
